##### Copyright 2023 The MediaPipe Authors. All Rights Reserved.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Holistic Landmarks Detection with MediaPipe Tasks

This notebook shows you how to use MediaPipe Tasks Python API to detect holistic landmarks from images.

## Preparation

Let's start with installing MediaPipe.

In [ ]:
!pip install -q mediapipe

Then download an off-the-shelf model bundle. Check out the [MediaPipe documentation](https://developers.google.com/mediapipe/solutions/vision/holistic_landmarker#models) for more information about this model bundle.

In [ ]:
!wget -O holistic_landmarker.task -q https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/1/holistic_landmarker.task

## Visualization utilities

In [ ]:
#@markdown We implemented some functions to visualize the holistic landmark detection results.
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles
import numpy as np
import matplotlib.pyplot as plt


def draw_landmarks_on_image(rgb_image, detection_result):
  pose_landmarks = detection_result.pose_landmarks
  face_landmarks = detection_result.face_landmarks
  left_hand_landmarks = detection_result.left_hand_landmarks
  right_hand_landmarks = detection_result.right_hand_landmarks
  annotated_image = np.copy(rgb_image[:, :, :3] if rgb_image.shape[2] == 4 else rgb_image)

  # Draw pose landmarks.
  if pose_landmarks:
    pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
    pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=pose_landmarks,
        connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
        landmark_drawing_spec=pose_landmark_style,
        connection_drawing_spec=pose_connection_style)

  # Draw face landmarks.
  if face_landmarks:
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=face_landmarks,
        connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
        landmark_drawing_spec=None,
        connection_drawing_spec=drawing_styles.get_default_face_mesh_tesselation_style())
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=face_landmarks,
        connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
        landmark_drawing_spec=None,
        connection_drawing_spec=drawing_styles.get_default_face_mesh_contours_style())
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=face_landmarks,
        connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_LEFT_IRIS,
        landmark_drawing_spec=None,
        connection_drawing_spec=drawing_styles.get_default_face_mesh_iris_connections_style())
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=face_landmarks,
        connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_RIGHT_IRIS,
        landmark_drawing_spec=None,
        connection_drawing_spec=drawing_styles.get_default_face_mesh_iris_connections_style())

  # Draw hand landmarks.
  if left_hand_landmarks:
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=left_hand_landmarks,
        connections=vision.HandLandmarksConnections.HAND_CONNECTIONS,
        landmark_drawing_spec=drawing_styles.get_default_hand_landmarks_style(),
        connection_drawing_spec=drawing_styles.get_default_hand_connections_style())

  if right_hand_landmarks:
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=right_hand_landmarks,
        connections=vision.HandLandmarksConnections.HAND_CONNECTIONS,
        landmark_drawing_spec=drawing_styles.get_default_hand_landmarks_style(),
        connection_drawing_spec=drawing_styles.get_default_hand_connections_style())

  return annotated_image


def plot_face_blendshapes_bar_graph(face_blendshapes):
  # Extract the face blendshapes category names and scores.
  face_blendshapes_names = [face_blendshapes_category.category_name for face_blendshapes_category in face_blendshapes]
  face_blendshapes_scores = [face_blendshapes_category.score for face_blendshapes_category in face_blendshapes]
  # The blendshapes are ordered in decreasing score value.
  face_blendshapes_ranks = range(len(face_blendshapes_names))

  fig, ax = plt.subplots(figsize=(12, 12))
  bar = ax.barh(face_blendshapes_ranks, face_blendshapes_scores, label=[str(x) for x in face_blendshapes_ranks])
  ax.set_yticks(face_blendshapes_ranks, face_blendshapes_names)
  ax.invert_yaxis()

  # Label each bar with values
  for score, patch in zip(face_blendshapes_scores, bar.patches):
    plt.text(patch.get_x() + patch.get_width(), patch.get_y(), f"{score:.4f}", va="top")

  ax.set_xlabel('Score')
  ax.set_title("Face Blendshapes")
  plt.tight_layout()
  plt.show()

## Download test image

Let's grab a test image that we'll use later. The image is from [Unsplash](https://unsplash.com/photos/mt2fyrdXxzk).

In [ ]:
!wget -q -O image.png https://storage.googleapis.com/mediapipe-assets/business-person.png

import cv2
from google.colab.patches import cv2_imshow

img = cv2.imread("image.png")
h, w, _ = img.shape
# Resize image to fit on screen
DESIRED_HEIGHT = 480
new_h = DESIRED_HEIGHT
new_w = int(round(w * (DESIRED_HEIGHT / h) / 16.0)) * 16
img = cv2.resize(img, (new_w, new_h))
cv2.imwrite("image.png", img)

cv2_imshow(img)

Optionally, you can upload your own image. If you want to do so, uncomment and run the cell below.

In [ ]:
# from google.colab import files
# uploaded = files.upload()

# for filename in uploaded:
#   content = uploaded[filename]
#   with open(filename, 'wb') as f:
#     f.write(content)

# if len(uploaded.keys()):
#   IMAGE_FILE = next(iter(uploaded))
#   print('Uploaded file:', IMAGE_FILE)
#   img = cv2.imread(IMAGE_FILE)
#   h, w, _ = img.shape
#   DESIRED_HEIGHT = 480
#   new_h = DESIRED_HEIGHT
#   new_w = int(round(w * (DESIRED_HEIGHT / h) / 16.0)) * 16
#   img = cv2.resize(img, (new_w, new_h))
#   cv2.imwrite(IMAGE_FILE, img)

## Running inference and visualizing the results

Here are the steps to run holistic landmark detection using MediaPipe.

Check out the [MediaPipe documentation](https://developers.google.com/mediapipe/solutions/vision/holistic_landmarker/python) to learn more about configuration options that this task supports.


In [ ]:
# STEP 1: Import the necessary modules.
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
from google.colab.patches import cv2_imshow

# STEP 2: Create a HolisticLandmarker object.
base_options = python.BaseOptions(model_asset_path='holistic_landmarker.task')
options = vision.HolisticLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    output_segmentation_mask=True)
detector = vision.HolisticLandmarker.create_from_options(options)

# STEP 3: Load the input image.
image = mp.Image.create_from_file("image.png")

# STEP 4: Detect holistic landmarks from the input image.
detection_result = detector.detect(image)

# STEP 5: Process the detection result. In this case, visualize it.
annotated_image = draw_landmarks_on_image(image.numpy_view(), detection_result)
cv2_imshow(cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR))

We will also visualize the face blendshapes categories using a bar graph.

In [ ]:
if detection_result.face_blendshapes:
  plot_face_blendshapes_bar_graph(detection_result.face_blendshapes)

Visualize the pose segmentation mask.

In [ ]:
if detection_result.segmentation_mask is not None:
  segmentation_mask = detection_result.segmentation_mask.numpy_view()
  segmentation_mask = np.squeeze(segmentation_mask)

  # Convert to 3‑channel uint8 image for visualization.
  visualized_mask = (segmentation_mask * 255).astype(np.uint8)
  visualized_mask = np.stack([visualized_mask]*3, axis=-1)
  cv2_imshow(visualized_mask)